# AoC 2024 Day 3 — Mull It Over

**Spark — regex extraction + a stateful window**

Puzzle: <https://adventofcode.com/2024/day/3>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

---

## The puzzle

A block of corrupted memory containing valid `mul(X,Y)` instructions buried in garbage.

- **Part 1** — find every *valid* `mul(X,Y)` (1–3 digits each), multiply, and sum.
- **Part 2** — `do()` and `don't()` instructions toggle whether the muls that follow count. Multiplication starts **enabled**. Sum only the enabled ones.

## The approach

Part 2 is the one worth the trip. "Scan left to right carrying a flag" sounds hopelessly sequential, and it is exactly where people reach for a UDF or `collect()`.

It is really a **last-non-null-value window** — one of the most reusable patterns in Spark:

```sql
last(flag, ignoreNulls => true) OVER (
  ORDER BY pos ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
```

`mul` tokens carry no flag (null); `do()`/`don't()` carry 1/0. The window fills each mul with the most recent toggle. `coalesce(..., 1)` covers the muls that appear before any toggle, since multiplication starts enabled.

The ordering key comes free: `regexp_extract_all` returns matches **in source order**, so `posexplode` gives a reliable position without computing character offsets.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day03

spark = get_spark('aoc-2024-day03')
print('Spark', spark.version)

## The published example

Day 3 publishes **two** examples — the part 1 one has no `do()`/`don't()` toggles, so part 2 needs its own.

In [ ]:
EXAMPLE_PART1 = r"xmul(2,4)%&mul[3,7]!@^do_not_mul(5,5)+mul(32,64]then(mul(11,8)mul(8,5))"
EXAMPLE_PART2 = r"xmul(2,4)&mul[3,7]!^don't()_mul(5,5)+mul(32,64](mul(11,8)undo()?mul(8,5))"

print('part 1:', day03.part1(spark, EXAMPLE_PART1), '(expected 161)')
print('part 2:', day03.part2(spark, EXAMPLE_PART2), '(expected 48)')

### The window doing the work

This is the whole puzzle in one table — read down the `enabled` column and watch it carry forward from each toggle.

In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

memory = spark.createDataFrame([(EXAMPLE_PART2,)], 'memory STRING')
tokens = memory.select(
    F.posexplode(
        F.regexp_extract_all(F.col('memory'), F.lit(day03.TOKEN), F.lit(0))
    ).alias('pos', 'token')
)

toggle = (
    F.when(F.col('token') == 'do()', F.lit(1))
    .when(F.col('token') == "don't()", F.lit(0))
    .otherwise(F.lit(None))
)
running = Window.orderBy('pos').rowsBetween(Window.unboundedPreceding, Window.currentRow)

tokens.select(
    'pos',
    'token',
    toggle.alias('toggle'),
    F.coalesce(F.last(toggle, ignorenulls=True).over(running), F.lit(1)).alias('enabled'),
).show(truncate=False)

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 3)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

for part in (1, 2):
    fn = getattr(day03, f'part{part}')
    started = time.perf_counter()
    answer = fn(spark, data)
    print(f'part {part}: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Notes & gotchas

- Part 1 and part 2 use **different published examples** — the part 1 example has no toggles. Running part 2 on the part 1 example gives 161, not 48.
- The whole input is loaded as a **single row**, not one row per line: instructions run across line breaks, so splitting on newlines would corrupt tokens that straddle them.
- This is the day where a single-partition window is genuinely load-bearing. A partitioned window would reset the enabled state at each partition boundary and quietly produce a wrong answer.